<a href="https://colab.research.google.com/github/ZaidKhan2002/GenAI_Experimentation/blob/main/NS_Code_Session_3_3_Optional_Extras_With_RAGAS_LLM_Metric_Definition.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# RAG Evaluation with Local Metrics, Custom LLM Judge, and RAGAS

This notebook evaluates the same local retrieval system in three stages:

1. Local non-LLM metrics
2. A custom GPT-4o-mini LLM judge through OpenRouter
3. RAGAS LLM-based metrics through OpenRouter

The RAGAS section is intentionally placed after the custom judge so the difference
between a manually designed judge and a framework-managed evaluation is easy to see.


In [ ]:
!pip install -q -U sentence-transformers ragas pandas openai pydantic langchain_community rouge_score rapidfuzz

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 79.5/79.5 kB 4.5 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 40.9/40.9 kB 3.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 611.3/611.3 kB 22.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 466.5/466.5 kB 28.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.0/11.0 MB 93.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.4/2.4 MB 85.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.1/3.1 MB 88.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 45.5/45.5 kB 4.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 64.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 69.4/69.4 kB 5.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 73.1/73.1 kB 5.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 252.5/252.5 kB 17.5 MB/s eta 0:00:00
   ━━━━━━━━━

In [ ]:
import numpy as np
import pandas as pd
from sentence_transformers import SentenceTransformer

MODEL_NAME = "sentence-transformers/all-MiniLM-L6-v2"
model = SentenceTransformer(MODEL_NAME)

print("Loaded:", MODEL_NAME)

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

...

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Loaded: sentence-transformers/all-MiniLM-L6-v2


In [ ]:
audit_data = ['MLOps Audit Q4: European division legacy branches report a 15% OCR failure rate.',
              "Tuesday Review confirmed the 15% spike is due to 'Legacy Scan-X' hardware and firmware v2.1.",
              'Jaymin approved a $45,000 emergency budget to upgrade European scanners by Q1 end.',
              'OCR failures peak on Tuesdays due to weekly bulk-batch processing of handwritten PDFs.',
              'Tony recommends a distributed architecture for handling 500+ PDFs in legacy branches.',
              "The 15% error rate is classified as 'Critical' for Banking and Compliance audits.",
              "Marten's team is monitoring OCR logs 24/7 until the hardware upgrade is finished.",
              'European legacy branches are the only units still using the v2.1 firmware.',
              'The new firmware v3.0 has been successfully tested in the North American cluster.',
              'Budget allocation for Q1 also includes a 10% reserve for unexpected cloud egress costs.',
              'Anisha suggested moving OCR processing to an asynchronous queue using RabbitMQ.',
              'Legacy Scan-X machines have a known overheating issue when processing over 100 pages.',
              'Compliance team noted that OCR errors are leading to incorrect data in customer KYC files.',
              "The upgrade project is codenamed 'Project Vision' and is led by the MLOps team.",
              'The target OCR failure rate after the upgrade is below 2%.',
              'European branches process approximately 12,000 scanned documents every week.',
              'Project Vision includes replacing scanners, upgrading firmware, and improving monitoring.',
              "The hardware vendor 'OptiScan' has been notified about the hardware failures.",
              'A temporary patch was deployed on Monday to reduce memory leaks during batch processing.',
              'Q2 Roadmap: Complete migration of all legacy branches to the centralized MLOps platform.']

# Normalize embeddings so dot product becomes cosine similarity.
document_embeddings = model.encode(audit_data, normalize_embeddings=True)

print("Documents:", len(audit_data))
print("Embedding shape:", document_embeddings.shape)

Documents: 20
Embedding shape: (20, 384)


In [ ]:
def retrieve(question, top_k=3):
    question_embedding = model.encode([question],normalize_embeddings=True)[0]

    scores = document_embeddings @ question_embedding
    top_indices = np.argsort(scores)[::-1][:top_k]

    return [
        {
            "text": audit_data[index],
            "score": float(scores[index])
        }
        for index in top_indices
    ]


def generate_answer(question):
    """Simple local extractive generation: return the best retrieved sentence."""
    results = retrieve(question, top_k=3)
    answer = results[0]["text"]  # usually llm combined answer
    contexts = [item["text"] for item in results]
    return answer, contexts


question = "Why do OCR failures peak on Tuesdays?"
answer, contexts = generate_answer(question)

print("Question:", question)
print("Answer:", answer)
print("\nRetrieved contexts:")
for context in contexts:
    print("-", context)

Question: Why do OCR failures peak on Tuesdays?
Answer: OCR failures peak on Tuesdays due to weekly bulk-batch processing of handwritten PDFs.

Retrieved contexts:
- OCR failures peak on Tuesdays due to weekly bulk-batch processing of handwritten PDFs.
- The target OCR failure rate after the upgrade is below 2%.
- MLOps Audit Q4: European division legacy branches report a 15% OCR failure rate.


In [ ]:
# Ground-truth evaluation dataset
golden_dataset = [
    {
        "question": "What is the OCR failure rate in European legacy branches?",
        "ground_truth": "The OCR failure rate is 15%.",
        "ground_truth_context": audit_data[0],
    },
    {
        "question": "Why do OCR failures peak on Tuesdays?",
        "ground_truth": "OCR failures peak because of weekly bulk-batch processing of handwritten PDFs.",
        "ground_truth_context": audit_data[3],
    },
    {
        "question": "What is the codename for the upgrade project?",
        "ground_truth": "The upgrade project is codenamed Project Vision.",
        "ground_truth_context": audit_data[13],
    },
]

evaluation_rows = []

for item in golden_dataset:
    answer, contexts = generate_answer(item["question"])

    evaluation_rows.append({
        "question": item["question"],
        "answer": answer,
        "ground_truth": item["ground_truth"],
        "contexts": contexts,
        "ground_truth_context": item["ground_truth_context"],
    })

evaluation_df = pd.DataFrame(evaluation_rows)
evaluation_df[["question", "answer", "ground_truth"]]

,question,answer,ground_truth
0,What is the OCR failure rate in European legac...,MLOps Audit Q4: European division legacy branc...,The OCR failure rate is 15%.
1,Why do OCR failures peak on Tuesdays?,OCR failures peak on Tuesdays due to weekly bu...,OCR failures peak because of weekly bulk-batch...
2,What is the codename for the upgrade project?,The upgrade project is codenamed 'Project Visi...,The upgrade project is codenamed Project Vision.


In [ ]:
import sys, types
fake_module = types.ModuleType("langchain_community.chat_models.vertexai")

class ChatVertexAI:  # dummy placeholder, never actually used
    pass

fake_module.ChatVertexAI = ChatVertexAI
sys.modules["langchain_community.chat_models.vertexai"] = fake_module

import ragas
print("RAGAS version:", ragas.__version__)

RAGAS version: 0.4.3


In [ ]:
from ragas.embeddings import HuggingFaceEmbeddings
from ragas.metrics.collections import (
    SemanticSimilarity,
    RougeScore,
    NonLLMStringSimilarity,
    DistanceMeasure,
)

# Local Sentence Transformer used by RAGAS.
ragas_embeddings = HuggingFaceEmbeddings(
    model=MODEL_NAME,
    normalize_embeddings=True,
)

semantic_metric = SemanticSimilarity(embeddings=ragas_embeddings)
rouge_metric = RougeScore()
string_metric = NonLLMStringSimilarity(
    distance_measure=DistanceMeasure.LEVENSHTEIN
)

metric_rows = []

for row in evaluation_rows:

    semantic_result = await semantic_metric.ascore(
        reference=row["ground_truth"],
        response=row["answer"],
    )

    rouge_result = await rouge_metric.ascore(
        reference=row["ground_truth"],
        response=row["answer"],
    )

    string_result = await string_metric.ascore(
        reference=row["ground_truth"],
        response=row["answer"],
    )

    semantic_score = semantic_result.value
    rouge_score = rouge_result.value
    string_score = string_result.value

    # Did the expected context appear in the retrieved contexts?
    context_hit = float(
        row["ground_truth_context"] in row["contexts"]
    )

    metric_rows.append({
        "question": row["question"],
        "answer": row["answer"],
        "ground_truth": row["ground_truth"],
        "semantic_similarity": semantic_score,
        "rouge_l": rouge_score,
        "string_similarity": string_score,
        "context_hit": context_hit,
    })


ragas_results_df = pd.DataFrame(metric_rows)
display(ragas_results_df)

print("\nAverage scores:")

display(
    ragas_results_df[
        [
            "semantic_similarity",
            "rouge_l",
            "string_similarity",
            "context_hit",
        ]
    ].mean().to_frame("average_score")
)

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

,question,answer,ground_truth,semantic_similarity,rouge_l,string_similarity,context_hit
0,What is the OCR failure rate in European legac...,MLOps Audit Q4: European division legacy branc...,The OCR failure rate is 15%.,0.669888,0.315789,0.187500,1.0
1,Why do OCR failures peak on Tuesdays?,OCR failures peak on Tuesdays due to weekly bu...,OCR failures peak because of weekly bulk-batch...,0.984989,0.769231,0.848837,1.0
2,What is the codename for the upgrade project?,The upgrade project is codenamed 'Project Visi...,The upgrade project is codenamed Project Vision.,0.891279,0.666667,0.607595,1.0



Average scores:


,average_score
semantic_similarity,0.848718
rouge_l,0.583896
string_similarity,0.547977
context_hit,1.000000


# Non-LLM Evaluation Metrics: Simple Explanations and Examples

The following metrics compare the generated answer with the ground-truth answer without using an LLM judge.

These metrics are faster and cheaper than LLM-based evaluation metrics.

The code calculates four values:

1. Semantic similarity
2. ROUGE score
3. String similarity
4. Context hit

---

## 1. Semantic Similarity

Semantic similarity checks whether the generated answer and the ground-truth answer have similar meanings.

It uses the local `all-MiniLM-L6-v2` embedding model.

The embedding model converts both answers into numerical vectors. It then compares those vectors to determine how close their meanings are.

### Example

**Ground-truth answer:**

> The OCR failure rate is 15%.

**Generated answer:**

> Fifteen percent of the OCR scans fail.

The wording is different, but both sentences express the same meaning.

Therefore, the semantic-similarity score should be high.

Now consider:

> The scanner was installed last year.

This sentence has a different meaning from the ground-truth answer.

Therefore, the semantic-similarity score should be low.

### Interpretation

* A score close to `1` means the meanings are very similar.
* A lower score means the meanings are less similar.
* Because the embeddings are normalized, the comparison behaves like cosine similarity.

Semantic similarity is useful when the generated answer uses different wording but communicates the correct idea.

---

## 2. ROUGE Score

ROUGE checks how many words or sequences of words are shared between the generated answer and the ground-truth answer.

It focuses more on text overlap than meaning.

In this code, `RougeScore()` uses the default ROUGE configuration provided by the installed RAGAS version.

ROUGE-L commonly compares the longest sequence of words that appears in both texts in the same order.

The words do not always need to appear next to each other, but their order should be preserved.

### Example

**Ground-truth answer:**

> The OCR failure rate is 15 percent.

**Generated answer:**

> The OCR failure rate is 15 percent in Europe.

Most of the words are shared and appear in the same order.

Therefore, the ROUGE score should be high.

Now consider:

> Fifteen out of every hundred document scans fail.

This answer has a similar meaning, but it uses different words.

Its semantic-similarity score may be high, while its ROUGE score may be lower because there is less direct word overlap.

### Interpretation

* A high ROUGE score means the generated answer shares many words or word sequences with the reference answer.
* A low ROUGE score means there is little direct textual overlap.
* A low ROUGE score does not always mean the answer is incorrect.

ROUGE is useful when the wording of the generated answer is expected to be close to the reference answer.

---

## 3. String Similarity

String similarity checks how similar the actual characters in the generated answer are to the characters in the ground-truth answer.

The code uses the Levenshtein distance.

Levenshtein distance counts the minimum number of character-level changes needed to convert one string into another.

The possible changes are:

* Inserting a character
* Deleting a character
* Replacing a character

RAGAS converts this distance into a similarity score.

### Example

**Ground-truth answer:**

> OCR failure rate is 15%.

**Generated answer:**

> OCR failure rate is 16%.

Only one character is different: `5` was replaced with `6`.

The two strings are therefore very similar at the character level, even though the numerical fact is incorrect.

Now consider:

> Weekly document processing causes delays.

This sentence is very different from the ground-truth answer.

Therefore, the string-similarity score should be low.

### Interpretation

* A score close to `1` means the two strings are almost identical.
* A lower score means more character changes are required.
* String similarity checks text structure, not factual correctness or meaning.

This metric is useful for checking spelling differences, formatting changes, short answers, codes, labels, names, or nearly identical text.

---

## 4. Context Hit

Context hit checks whether the expected ground-truth context appears in the retrieved contexts.

The code performs an exact membership check:

```python
row["ground_truth_context"] in row["contexts"]
```

If the exact expected context is present, the score is:

```text
1.0
```

If it is not present, the score is:

```text
0.0
```

### Example

**Expected ground-truth context:**

> The OCR failure rate is 15%.

**Retrieved contexts:**

1. The OCR failure rate is 15%.
2. European branches process handwritten documents.
3. Scanner maintenance occurs monthly.

The expected context appears in the retrieved list.

Therefore:

```text
context_hit = 1.0
```

Now suppose the retrieved contexts are:

1. OCR systems are used in European branches.
2. Scanner maintenance occurs monthly.
3. Bulk processing happens on Tuesdays.

The exact expected context is missing.

Therefore:

```text
context_hit = 0.0
```

### Important Limitation

This code checks for an exact match.

It may return `0.0` even when a retrieved context contains the same meaning with slightly different wording.

For example:

**Expected context:**

> The OCR failure rate is 15%.

**Retrieved context:**

> Fifteen percent of OCR scans fail.

These sentences mean the same thing, but the exact string is different.

Therefore, the current exact-match check may still return `0.0`.

---

## Comparison of the Metrics

| Metric              | What it compares                       | What it mainly checks                      |
| ------------------- | -------------------------------------- | ------------------------------------------ |
| Semantic Similarity | Generated answer vs ground truth       | Similarity in meaning                      |
| ROUGE               | Generated answer vs ground truth       | Word and phrase overlap                    |
| String Similarity   | Generated answer vs ground truth       | Character-level similarity                 |
| Context Hit         | Expected context vs retrieved contexts | Whether the expected context was retrieved |

---

## Example Comparison

**Ground truth:**

> The OCR failure rate is 15%.

**Generated answer:**

> Fifteen percent of OCR scans fail.

The likely results are:

* **Semantic similarity:** High, because the meanings are similar.
* **ROUGE:** Moderate, because some words are different.
* **String similarity:** Moderate or low, because the character sequences differ.
* **Context hit:** Depends only on whether the expected context appears in the retrieved list.

Now consider:

> The OCR failure rate is 16%.

The likely results are:

* **Semantic similarity:** High, because the topic and sentence meaning are similar.
* **ROUGE:** High, because most words match.
* **String similarity:** Very high, because only one character changed.

This shows why one metric should not be used alone.

---

## Understanding the Average Scores

After calculating the metrics for every evaluation row, the code finds the average value for each metric.

For example:

```text
semantic_similarity    0.86
rouge_l                 0.72
string_similarity       0.68
context_hit             0.80
```

This can be interpreted as follows:

* The generated answers are generally close in meaning to the reference answers.
* There is a reasonable amount of word overlap.
* The exact wording differs more than the meaning.
* The expected context was retrieved for 80% of the questions.

For `context_hit`, the average has a direct meaning.

An average context-hit score of `0.80` means the expected context was found in 80% of the evaluation examples.

---

## Simple Summary

* **Semantic similarity** checks whether two answers mean the same thing.
* **ROUGE** checks whether they use similar words and word sequences.
* **String similarity** checks whether their characters and exact wording are similar.
* **Context hit** checks whether the expected source context was retrieved.

These metrics do not fully determine whether an answer is factually correct.

For reliable RAG evaluation, they are best used together with LLM-based metrics such as faithfulness, answer correctness, context precision, and context recall.


In [ ]:
import os
from getpass import getpass
from openai import OpenAI, AsyncOpenAI

# Enter the key securely when this cell runs.
os.environ["OPENROUTER_API_KEY"] = getpass(
        "Enter your OpenRouter API key: "
    )

# openrouter_client = OpenAI(
#     api_key=os.environ["OPENROUTER_API_KEY"],
#     base_url="https://openrouter.ai/api/v1",
# )


openrouter_client = AsyncOpenAI(
    api_key=os.environ["OPENROUTER_API_KEY"],
    base_url="https://openrouter.ai/api/v1",
)

JUDGE_MODEL = "openai/gpt-4o-mini"
print("Judge model:", JUDGE_MODEL)

Enter your OpenRouter API key: ··········
Judge model: openai/gpt-4o-mini


## Custom LLM-as-a-Judge with OpenRouter GPT-4o Mini

The section above uses RAGAS-defined prompts and scoring procedures. This section uses a manually written prompt so you can compare a custom judge with RAGAS.

The custom judge scores:

- **Correctness**
- **Relevance**
- **Faithfulness**

It reuses the same OpenRouter client and model.

In [ ]:

from pydantic import BaseModel, Field, ConfigDict

class JudgeResult(BaseModel):
    # Prevent the judge from returning fields outside this schema.
    model_config = ConfigDict(extra="forbid")

    correctness: float = Field(ge=0, le=1)
    relevance: float = Field(ge=0, le=1)
    faithfulness: float = Field(ge=0, le=1)
    reason: str


def llm_judge(question, answer, ground_truth, contexts):
    context_text = "\n".join(f"{number}. {context}"
        for number, context in enumerate(contexts, start=1)
    )

    prompt = f"""
Evaluate the RAG answer using scores from 0 to 1.

Scoring rules:
- correctness: agreement with the ground-truth answer
- relevance: how directly the answer addresses the question
- faithfulness: whether every claim in the answer is supported by the retrieved contexts

Question:
{question}

Generated answer:
{answer}

Ground-truth answer:
{ground_truth}

Retrieved contexts:
{context_text}

Give strict scores. Briefly explain the main reason.
"""

    response = openrouter_client.chat.completions.create(
        model=JUDGE_MODEL,
        messages=[
            {
                "role": "system",
                "content": (
                    "You are a strict evaluator of RAG systems. "
                    "Return only data matching the supplied JSON schema."
                ),
            },
            {"role": "user", "content": prompt},
        ],
        response_format={
            "type": "json_schema",
            "json_schema": {
                "name": "rag_judge_result",
                "strict": True,
                "schema": JudgeResult.model_json_schema(),
            },
        },
        temperature=0,
    )

    return JudgeResult.model_validate_json(
        response.choices[0].message.content
    )

In [ ]:
judge_rows = []

for row in evaluation_rows:
    judge = llm_judge(
        question=row["question"],
        answer=row["answer"],
        ground_truth=row["ground_truth"],
        contexts=row["contexts"],
    )

    judge_rows.append({
        "question": row["question"],
        "llm_correctness": judge.correctness,
        "llm_relevance": judge.relevance,
        "llm_faithfulness": judge.faithfulness,
        "llm_reason": judge.reason,
    })

judge_df = pd.DataFrame(judge_rows)
display(judge_df)

,question,llm_correctness,llm_relevance,llm_faithfulness,llm_reason
0,What is the OCR failure rate in European legac...,1.0,1.0,1.0,The generated answer is fully correct as it ma...
1,Why do OCR failures peak on Tuesdays?,1.0,1.0,1.0,The generated answer is fully aligned with the...
2,What is the codename for the upgrade project?,1.0,1.0,1.0,The generated answer is fully correct as it ma...


In [ ]:
# Join local metrics, RAGAS LLM metrics, and custom-judge scores.
combined_results_df = ragas_results_df.merge(
    judge_df,
    on="question",
    how="left",
)

display(combined_results_df)

print("\nCustom LLM-judge average scores:")
display(
    combined_results_df[
        [
            "llm_correctness",
            "llm_relevance",
            "llm_faithfulness",
        ]
    ].mean().to_frame("average_score")
)

,question,answer,ground_truth,semantic_similarity,rouge_l,string_similarity,context_hit,llm_correctness,llm_relevance,llm_faithfulness,llm_reason
0,What is the OCR failure rate in European legac...,MLOps Audit Q4: European division legacy branc...,The OCR failure rate is 15%.,0.669888,0.315789,0.187500,1.0,1.0,1.0,1.0,The generated answer is fully correct as it ma...
1,Why do OCR failures peak on Tuesdays?,OCR failures peak on Tuesdays due to weekly bu...,OCR failures peak because of weekly bulk-batch...,0.984989,0.769231,0.848837,1.0,1.0,1.0,1.0,The generated answer is fully aligned with the...
2,What is the codename for the upgrade project?,The upgrade project is codenamed 'Project Visi...,The upgrade project is codenamed Project Vision.,0.891279,0.666667,0.607595,1.0,1.0,1.0,1.0,The generated answer is fully correct as it ma...



Custom LLM-judge average scores:


,average_score
llm_correctness,1.0
llm_relevance,1.0
llm_faithfulness,1.0


### Reading the LLM-judge scores

All three scores range from **0 to 1**:

- A score near **1** is strong.
- A score near **0** is weak.
- `llm_reason` explains why the judge selected those scores.

LLM judging is more flexible than ROUGE or string matching, but it is also model-dependent and costs API tokens. For dependable evaluation, combine LLM-judge scores with deterministic metrics and human review.

The notebook now contains three layers of evaluation:

1. Local deterministic metrics
2. RAGAS LLM-based metrics
3. A custom LLM judge

Comparing them helps show how standardized RAGAS evaluation differs from a manually designed rubric.

## RAGAS LLM-Based Metrics with OpenRouter GPT-4o Mini

This section uses RAGAS's own metric implementations and prompts rather than a manually written judge prompt.

- **Faithfulness:** Are the claims in the answer supported by the retrieved contexts?
- **Answer relevancy:** Does the answer directly address the question?
- **Answer correctness:** Does the answer agree with the ground-truth answer?
- **Context precision:** Are useful retrieved chunks ranked ahead of irrelevant chunks?
- **Context recall:** Do the retrieved chunks contain the information needed by the ground truth?

RAGAS uses GPT-4o mini for the reasoning steps. `all-MiniLM-L6-v2` remains the embedding model for answer relevancy and answer correctness.

In [ ]:
from ragas.llms import llm_factory
from ragas.metrics.collections import (
    Faithfulness,
    AnswerRelevancy,
    AnswerCorrectness,
    ContextPrecision,
    ContextRecall,
)

# Wrap the OpenRouter OpenAI-compatible client for RAGAS.
ragas_judge_llm = llm_factory(
    JUDGE_MODEL,
    provider="openai",
    client=openrouter_client,
)

# Metrics that only need an LLM.
faithfulness_metric = Faithfulness(llm=ragas_judge_llm)
context_precision_metric = ContextPrecision(llm=ragas_judge_llm)
context_recall_metric = ContextRecall(llm=ragas_judge_llm)

# These metrics use both an LLM and embeddings.
answer_relevancy_metric = AnswerRelevancy(
    llm=ragas_judge_llm,
    embeddings=ragas_embeddings,
)
answer_correctness_metric = AnswerCorrectness(
    llm=ragas_judge_llm,
    embeddings=ragas_embeddings,
)

print("RAGAS LLM metrics initialized")

RAGAS LLM metrics initialized


In [ ]:
ragas_llm_rows = []

for row in evaluation_rows:

    faithfulness_result = await faithfulness_metric.ascore(
        user_input=row["question"],
        response=row["answer"],
        retrieved_contexts=row["contexts"],
    )

    relevancy_result = await answer_relevancy_metric.ascore(
        user_input=row["question"],
        response=row["answer"],
    )

    correctness_result = await answer_correctness_metric.ascore(
        user_input=row["question"],
        response=row["answer"],
        reference=row["ground_truth"],
    )

    context_precision_result = await context_precision_metric.ascore(
        user_input=row["question"],
        reference=row["ground_truth"],
        retrieved_contexts=row["contexts"],
    )

    context_recall_result = await context_recall_metric.ascore(
        user_input=row["question"],
        reference=row["ground_truth"],
        retrieved_contexts=row["contexts"],
    )

    ragas_llm_rows.append({
        "question": row["question"],
        "ragas_faithfulness": faithfulness_result.value,
        "ragas_answer_relevancy": relevancy_result.value,
        "ragas_answer_correctness": correctness_result.value,
        "ragas_context_precision": context_precision_result.value,
        "ragas_context_recall": context_recall_result.value,

    })


ragas_llm_df = pd.DataFrame(ragas_llm_rows)

display(ragas_llm_df)

print("\nAverage RAGAS LLM metric scores:")

display(
    ragas_llm_df[
        [
            "ragas_faithfulness",
            "ragas_answer_relevancy",
            "ragas_answer_correctness",
            "ragas_context_precision",
            "ragas_context_recall",
        ]
    ]
    .mean()
    .to_frame("average_score")
)

,question,ragas_faithfulness,ragas_answer_relevancy,ragas_answer_correctness,ragas_context_precision,ragas_context_recall
0,What is the OCR failure rate in European legac...,1.0,0.886595,0.667472,1.0,1.0
1,Why do OCR failures peak on Tuesdays?,1.0,0.868160,0.996247,1.0,1.0
2,What is the codename for the upgrade project?,1.0,0.929358,0.722820,1.0,1.0



Average RAGAS LLM metric scores:


,average_score
ragas_faithfulness,1.000000
ragas_answer_relevancy,0.894704
ragas_answer_correctness,0.795513
ragas_context_precision,1.000000
ragas_context_recall,1.000000


In [ ]:
# Combine local metrics and RAGAS LLM-based metrics.
ragas_results_df = combined_results_df.merge(
    ragas_llm_df,
    on="question",
    how="left",
)

display(ragas_results_df)

,question,answer,ground_truth,semantic_similarity,rouge_l,string_similarity,context_hit,llm_correctness,llm_relevance,llm_faithfulness,llm_reason,ragas_faithfulness,ragas_answer_relevancy,ragas_answer_correctness,ragas_context_precision,ragas_context_recall
0,What is the OCR failure rate in European legac...,MLOps Audit Q4: European division legacy branc...,The OCR failure rate is 15%.,0.669888,0.315789,0.187500,1.0,1.0,1.0,1.0,The generated answer is fully correct as it ma...,1.0,0.886595,0.667472,1.0,1.0
1,Why do OCR failures peak on Tuesdays?,OCR failures peak on Tuesdays due to weekly bu...,OCR failures peak because of weekly bulk-batch...,0.984989,0.769231,0.848837,1.0,1.0,1.0,1.0,The generated answer is fully aligned with the...,1.0,0.868160,0.996247,1.0,1.0
2,What is the codename for the upgrade project?,The upgrade project is codenamed 'Project Visi...,The upgrade project is codenamed Project Vision.,0.891279,0.666667,0.607595,1.0,1.0,1.0,1.0,The generated answer is fully correct as it ma...,1.0,0.929358,0.722820,1.0,1.0


# RAGAS LLM Metrics: Simple Explanations and Examples

The following RAGAS metrics use GPT-4o mini as the judge model. Some metrics also use the local `all-MiniLM-L6-v2` embedding model.

---

## 1. Faithfulness

Faithfulness checks whether the generated answer is supported by the retrieved context.

It examines the claims made in the answer and checks whether each claim can be supported using the retrieved documents.

### Example

**Retrieved context:**

> The OCR failure rate in European legacy branches is 15%.

**Generated answer:**

> The OCR failure rate is 15%, and the hardware upgrade was completed last week.

The first statement is supported by the retrieved context.

The second statement about the hardware upgrade is not mentioned in the context.

Therefore, the answer is only partially faithful.

* A high faithfulness score means that most of the answer is supported by the retrieved information.
* A low faithfulness score means that the answer contains unsupported or hallucinated information.

---

## 2. Answer Relevancy

Answer relevancy checks whether the generated answer directly responds to the user's question.

RAGAS examines the answer and tries to understand what question the answer is responding to. It then compares that reconstructed question with the original question.

The embedding model measures how similar the meanings of the two questions are.

### Example

**Original question:**

> What is the OCR failure rate?

**Generated answer:**

> The OCR failure rate is 15%.

This answer directly responds to the question, so the answer-relevancy score should be high.

Now consider this answer:

> The scanner firmware is version 2.1.

This statement may be correct, but it does not answer the question about the OCR failure rate.

Therefore, its answer-relevancy score should be low.

* A high score means the response is focused and directly answers the question.
* A low score means the response is unrelated or focused on a different topic.

---

## 3. Answer Correctness

Answer correctness checks whether the generated answer agrees with the expected or ground-truth answer.

It generally considers two things:

1. Whether the facts in the generated answer are correct.
2. Whether the overall meaning is similar to the reference answer.

The judge model checks which facts are correct, incorrect, extra, or missing.

The embedding model checks whether the generated answer and the reference answer have similar meanings, even when they use different words.

### Example

**Ground-truth answer:**

> The OCR failure rate in European legacy branches is 15%.

**Generated answer:**

> European legacy branches have a fifteen percent OCR error rate.

The wording is different, but both answers communicate the same fact.

Therefore, the answer-correctness score should be high.

Now consider:

> The OCR failure rate is 25%.

This answer discusses the correct topic, but the value is incorrect.

Therefore, the factual correctness score would be low.

Answer correctness is useful when a reference answer is available and you want to check how accurately the model reproduced it.

---

## 4. Context Precision

Context precision checks whether the retrieved documents are relevant to the question.

It also checks whether the most useful documents appear near the top of the retrieved results.

Relevant documents appearing at the beginning are considered more valuable than relevant documents appearing near the bottom.

### Example

Suppose three contexts are retrieved:

1. A relevant context about the OCR failure rate.
2. An irrelevant context about scanner installation.
3. Another relevant context about OCR errors.

This retrieval result is reasonably good because two of the three contexts are relevant.

However, one irrelevant context appears before the second useful context.

Therefore, the context-precision score would be good but not perfect.

Now suppose the results are:

1. Relevant context.
2. Relevant context.
3. Irrelevant context.

This ordering would receive a higher context-precision score because the useful information appears first.

* A high context-precision score means the retriever returned mostly useful documents and ranked them well.
* A low score means many retrieved documents were irrelevant or useful documents appeared too far down the list.

---

## 5. Context Recall

Context recall checks whether the retrieved documents contain all the information required to produce the correct answer.

It focuses on whether important information was missed during retrieval.

### Example

**Ground-truth answer:**

> OCR failures peak on Tuesdays because of weekly bulk-batch processing of handwritten PDFs.

This answer contains three important pieces of information:

1. OCR failures peak on Tuesdays.
2. The reason is weekly bulk-batch processing.
3. The files are handwritten PDFs.

Suppose the retrieved contexts mention that failures peak on Tuesdays and that bulk processing is the cause, but they do not mention handwritten PDFs.

The retrieval system captured most of the required information, but it missed one important detail.

Therefore, the context-recall score would be moderate rather than perfect.

* A high context-recall score means the retrieval system found nearly all the information needed for the correct answer.
* A low score means important supporting information was not retrieved.

---

## Comparison of the Metrics

| Metric             | What it checks                                                      |
| ------------------ | ------------------------------------------------------------------- |
| Faithfulness       | Whether the answer is supported by the retrieved contexts           |
| Answer Relevancy   | Whether the answer directly responds to the user's question         |
| Answer Correctness | Whether the answer matches the expected ground-truth answer         |
| Context Precision  | Whether the retrieved contexts are relevant and correctly ranked    |
| Context Recall     | Whether the retrieved contexts contain all the required information |

---

## Simple Summary

* **Faithfulness** checks whether the answer contains hallucinations.
* **Answer Relevancy** checks whether the model answered the actual question.
* **Answer Correctness** checks whether the answer is factually correct.
* **Context Precision** checks whether the retrieved documents are useful and properly ranked.
* **Context Recall** checks whether retrieval missed any important information.

A custom LLM judge and RAGAS may produce different scores because they use different evaluation instructions.

A custom judge follows the manually written scoring prompt, while RAGAS uses a separate evaluation procedure designed for each metric.
